# 08: As peças de busca do Investigator

O Investigator existe para explicar o **porquê** de um número estranho, não para repetir o número. Ele não é
mais um agente que escolhe tools: é um subgrafo de pesquisa com o caminho fixo em código (notebook 20). Duas
peças trazem os dados que ele usa, e as duas são chamadas **pelo código** do subgrafo, nunca pelo LLM:

- `buscar_historico_trimestral` (nó `preparar`): 8 trimestres de lucro, receita e lucro bruto, lidos do
  yfinance. Sem LLM e sem busca: serve para ver se a queda é de um trimestre só ou uma tendência.
- `pesquisar_pergunta` (nó `buscar`): uma busca na web para cada pergunta causal que o LLM escreveu. As
  perguntas da mesma rodada rodam em paralelo.

A tool `buscar_fatos_relevantes` continua no pacote (com os testes dela), mas nenhum agente a usa: as perguntas
do subgrafo já cobrem fatos relevantes e impairments, cada uma com o foco que a anomalia pede.

Este notebook mostra as duas peças com a VALE3.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Histórico trimestral

Um **trimestre** é o período de 3 meses em que as empresas abertas divulgam resultados. Os números estão em
bilhões, **na moeda em que a empresa publica os demonstrativos**: a VALE3 publica em dólar, então a tabela
vem em US\$. A tool lê a moeda da própria série do Yahoo e escreve o símbolo no cabeçalho.

- **Net Income (lucro líquido):** o que sobra depois de todos os custos, despesas, juros e impostos.
- **Total Revenue (receita):** o total vendido no período.
- **Gross Profit (lucro bruto):** receita menos o custo direto do que foi vendido.

In [2]:
from fluxo_financeiro.tools.investigation import buscar_historico_trimestral, pesquisar_pergunta

print(buscar_historico_trimestral.invoke("VALE3"))

2026-11-09 13:48:08,147 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ticker=VALE3.SA


2026-11-09 13:48:09,132 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ok — 7 trimestres


Histórico Trimestral de VALE3.SA (últimos 7 trimestres):
Valores em bilhões de US$ (moeda dos demonstrativos).

Métrica             2026-09-30    2026-06-30    2026-03-31    2025-12-31    2025-09-30    2025-06-30    2025-03-31    
----------------------------------------------------------------------------------------------------------------------
Net Income                1.25B        1.38B        1.89B       -3.84B        2.69B         N/A          N/A   
Total Revenue            10.60B       10.50B        9.26B       11.06B       10.42B         N/A          N/A   
Gross Profit              2.66B        3.20B        3.08B        4.28B        3.79B         N/A          N/A   



**Como ler a tabela** (a coluna mais à esquerda é o trimestre mais recente):
- **Pontual:** um trimestre fora da curva e os outros normais. Costuma ser um evento único, como uma baixa contábil.
- **Estrutural:** piora trimestre após trimestre. O negócio está perdendo força.
- **Cíclica:** sobe e desce acompanhando um ciclo externo, como o preço do minério de ferro.

Compare o lucro com a receita: se a receita ficou estável e o lucro despencou, o problema está nos custos ou num
evento contábil, não nas vendas.

## 2. Uma pergunta do subgrafo

O que o subgrafo procura na web:

- **Fato relevante:** comunicado que a empresa é obrigada a publicar na CVM quando acontece algo que pode mexer no preço da ação (uma venda de ativo, um acordo judicial, uma mudança na diretoria).
- **Impairment:** baixa contábil quando um ativo passa a valer menos do que está registrado no balanço. Derruba o lucro do trimestre sem tirar dinheiro do caixa.
- **Commodities e câmbio:** o preço do minério de ferro e o dólar mexem no lucro da Vale sem que nada mude dentro da empresa.

`pesquisar_pergunta(pergunta, ticker)` manda a pergunta à busca configurada (Google via Gemini por padrão; a
Tavily é a alternativa, notebook 21), com a instrução fixa de procurar a causa de uma variação atípica nos
últimos 12 meses. Como quem chama é o código, um erro temporário é repetido ali mesmo. Se a busca falhar de
vez ou voltar vazia, a função devolve `""`: a pergunta fica sem achado e as outras seguem.

Esta é a única chamada paga do notebook (uma busca).

In [3]:
achado = pesquisar_pergunta("O preço do minério de ferro explica a variação do lucro da Vale no último trimestre?", "VALE3")
print(achado or "(sem achado)")

2026-11-09 13:48:09,141 - fluxo_financeiro - INFO - [pesquisa] VALE3: O preço do minério de ferro explica a variação do lucro da Vale no último trimes


<conteudo_externo origem="google_search">
Sim, o preço do minério de ferro explica em parte a variação atípica do lucro da Vale (VALE3) no último trimestre reportado (3T26, divulgado em 29 de outubro de 2026), mas ele atuou em conjunto com outros fatores macroeconômicos e operacionais. No período, a mineradora registrou lucro líquido de US$ 1,255 bilhão, uma queda expressiva de 53% na base anual. O principal vetor dessa retração foi a redução de 10% no preço médio realizado dos finos de minério de ferro, que caiu para US$ 85,1 por tonelada, pressionando o Ebitda proforma do segmento a uma baixa de 20% (para US$ 3,5 bilhões).

Além da desvalorização do minério, o resultado foi fortemente impactado pela apreciação do real frente ao dólar — que elevou o custo-caixa C1 em 12%, para US$ 23,1 por tonelada — e pela alta nos preços do petróleo (bunker), que encareceu o frete marítimo. Por outro lado, a forte alta de 56% no preço realizado do cobre ajudou a mitigar perdas parciais por meio da d

## O que observar

- No histórico, veja se a queda do lucro é de um trimestre só ou de vários: é exatamente o que o Investigator precisa decidir.
- O cabeçalho da tabela diz a moeda dos demonstrativos. Para a VALE3 é US\$, mesmo com a ação cotada em reais na B3.
- Se o yfinance trouxer menos de 8 trimestres, a tabela mostra quantos vieram. O yfinance também atrasa os trimestres mais recentes.
- O achado chega dentro de `<conteudo_externo>`, e a origem da tag diz qual busca respondeu (`google_search` ou `tavily`): o texto é da web e o agente o trata como dado.
- A resposta termina com a linha `Fontes:`. O subgrafo junta as fontes da última linha `Fontes:` de cada achado no diagnóstico final.